# SC26 Tutorial — Session 2, Part 1: Linalg and tiling (CPU)

Session 1 built a matrix-matrix product from the bottom: `scf.for` loops written by hand, then a custom `blas.matmul` op with a pass that lowered it to those same loops.

This session starts one level higher, with ops from an existing dialect, and asks a new question: **same computation, same single thread — does changing the shape of the loops make it faster?**

1. **Linalg** — `linalg.matmul`, with `linalg.matvec` alongside as a second example: the `linalg.generic` form underneath them and the loop nests they stand for, lowered and JIT-executed as in session 1.
2. **Tiling** — split the loop nests into blocks with a **transform dialect schedule**: a small MLIR program whose input is your program. One schedule, written against an op *interface*, tiles both ops.
3. **Measuring it** — tiled matmul against untiled.

Part 2 takes the same tiles to the GPU.

**You write MLIR in a source file; every notebook cell is provided.** Everything here is CPU only, single-threaded.

## Setup

**Run the cells in order, starting here.** Later cells use names defined by earlier ones, so skipping ahead fails with `NameError`. `Ctrl+F8` (run every cell above) catches you up if you lose your place.

Same prebuilt MLIR toolchain as session 1.

In [ ]:
REPO = "MaheshRavishankar/sc26_tutorial"
RELEASE_TAG = "session1-toolchain-v1"
ASSET_NAME = "mlir-tutorial-llvm.tar.gz"

In [ ]:
import os
import requests

# Set MLIR_TUTORIAL_INSTALL_DIR to iterate locally (e.g. in VS Code) against
# an already-built toolchain -- skips the download entirely if it's already
# there. Unset in Colab, so this always defaults to the normal /content path.
INSTALL_DIR = os.environ.get("MLIR_TUTORIAL_INSTALL_DIR", "/content/mlir-tutorial-llvm")

if os.path.exists(f"{INSTALL_DIR}/bin/mlir-opt"):
    print(f"Using existing toolchain at {INSTALL_DIR}")
else:
    tar_path = f"{INSTALL_DIR}.tar.gz"
    url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(tar_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    !mkdir -p {INSTALL_DIR}
    !tar xf {tar_path} -C {INSTALL_DIR} --strip-components=1

In [ ]:
import os, sys

os.environ["PATH"] = f"{INSTALL_DIR}/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{INSTALL_DIR}/python_packages/mlir_core")

!mlir-opt --version

In [ ]:
from mlir.ir import Context, Module
from mlir.passmanager import PassManager
from mlir.execution_engine import ExecutionEngine
import mlir.runtime as rt
import ctypes
import numpy as np

print("MLIR Python bindings imported OK")

Also clone this repo -- every `.mlir` source below lives here.

In [ ]:
# Set MLIR_TUTORIAL_REPO_DIR to iterate locally against an already-cloned
# checkout -- skips the clone entirely if it's already there.
REPO_DIR = os.environ.get("MLIR_TUTORIAL_REPO_DIR", "/content/sc26_tutorial")
if os.path.isdir(REPO_DIR):
    print(f"Using existing repo checkout at {REPO_DIR}")
else:
    # --no-single-branch: every branch's tip, still at depth 1. It costs
    # nothing measurable, and it means the solution branches are already in
    # your checkout if you want a solved file out of one.
    !git clone --depth 1 --no-single-branch https://github.com/{REPO} {REPO_DIR}

## Step 1 — Linalg ops

In session 1 you wrote the loop nest yourself, then wrote a pass to generate it. Linalg is a third way: ops from an *existing* dialect that already mean "matrix multiply" and "matrix-vector product".

Two files, one op each, both complete — there is nothing to write in this step:

- **`session2/linalg_matmul.mlir`** — `@matmul`, the op this session is about;
- **`session2/linalg_matvec.mlir`** — `@matvec`, the same idea one dimension smaller, as a second example.

Note the accumulate: every Linalg named op on memrefs reads *and* writes its `outs` operand, so `linalg.matmul` computes `c += a @ b`. That is the same `+=` convention as session 1, and it is why `run_and_check` zeroes the output buffer before each call.

In [ ]:
# Opens the files. On Colab, files.view() puts them in the editor pane -- use it
# instead of the file browser, which refuses some file types. Running locally
# there is no google.colab, so this prints the paths instead and you open them
# in your own editor.
MATMUL_FILE = f"{REPO_DIR}/session2/linalg_matmul.mlir"
MATVEC_FILE = f"{REPO_DIR}/session2/linalg_matvec.mlir"

try:
    from google.colab import files

    files.view(MATMUL_FILE)
    files.view(MATVEC_FILE)
except ModuleNotFoundError:
    print(f"open in your editor:\n    {MATMUL_FILE}\n    {MATVEC_FILE}")

Round-trip both through `mlir-opt` first. No pass pipeline, so this only parses and re-prints each file.

In [ ]:
!mlir-opt {MATMUL_FILE}
assert _exit_code == 0
!mlir-opt {MATVEC_FILE}
assert _exit_code == 0

Before lowering anything, look at what the named ops actually *are*. `linalg-generalize-named-ops` rewrites each into `linalg.generic`, the op every Linalg named op is shorthand for:

- **`iterator_types`** — how many loops there are, and which are `parallel` and which is a `reduction`. Matmul has three: `i` and `j` parallel, `k` the reduction. Matvec has two: `i` parallel, `j` the reduction.
- **`indexing_maps`** — one per operand, saying which loop indices address it. For matmul, `a` is read at `(i, k)`, `b` at `(k, j)`, and `c` at `(i, j)`.
- **the body** — the scalar computation for one iteration: here a `mulf`, then an `addf` into the output element.

Every Linalg op implements an **op interface**, `LinalgOp`, that reports exactly this description, whichever op it is. Step 2 relies on that.

In [ ]:
!mlir-opt {MATMUL_FILE} --linalg-generalize-named-ops
assert _exit_code == 0
!mlir-opt {MATVEC_FILE} --linalg-generalize-named-ops
assert _exit_code == 0

Now lower them. `convert-linalg-to-loops` replaces each op with the loop nest it stands for: one `scf.for` per iterator type, a `memref.load` of each operand at the indices its indexing map selects, the body, and a `memref.store` back into the output. Compare `@matmul` against the `matmul_scf.mlir` you wrote by hand in session 1 — it is the same nest, generated rather than typed.

In [ ]:
!mlir-opt {MATMUL_FILE} --convert-linalg-to-loops
assert _exit_code == 0
!mlir-opt {MATVEC_FILE} --convert-linalg-to-loops
assert _exit_code == 0

Lower the rest of the way to the LLVM dialect and JIT `@matmul`, checking it against numpy. The pipeline is session 1's seven passes with `convert-linalg-to-loops` in front of them: it does the job your `convert-blas-to-scf` pass did there.

In [ ]:
CPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(llvm-request-c-wrappers),"
    "func.func(convert-linalg-to-loops),"
    "convert-scf-to-cf,"
    "convert-cf-to-llvm,"
    "convert-arith-to-llvm,"
    "finalize-memref-to-llvm,"
    "convert-func-to-llvm,"
    "reconcile-unrealized-casts"
    ")"
)

M, N, K = 128, 256, 512  # c is M x N; a is M x K; b is K x N


def memref_args(*arrays):
    return [ctypes.pointer(ctypes.pointer(rt.get_ranked_memref_descriptor(x))) for x in arrays]


def compile_file(path, **engine_options):
    """Parse an MLIR file, lower it to LLVM, and JIT-compile it."""
    ctx = Context()
    module = Module.parseFile(path, ctx)
    PassManager.parse(CPU_LOWERING_PIPELINE, ctx).run(module.operation)
    return ExecutionEngine(module, opt_level=3, **engine_options)


def run_and_check(engine, args, expected):
    """Call `matmul` on numpy arrays passed as memrefs. The last one is the
    output buffer, which linalg.matmul accumulates into, so it is zeroed
    first."""
    args[-1][...] = 0
    engine.invoke("matmul", *memref_args(*args))
    err = float(np.max(np.abs(args[-1] - expected)))
    print(f"result {args[-1].shape}, max abs error vs numpy = {err:.3g}")
    np.testing.assert_allclose(args[-1], expected, rtol=1e-3, atol=1e-3)
    print("OK")


rng = np.random.default_rng(0)
a = rng.random((M, K), dtype=np.float32)
b = rng.random((K, N), dtype=np.float32)
c = np.zeros((M, N), dtype=np.float32)
expected = a @ b

untiled_matmul = compile_file(MATMUL_FILE)
run_and_check(untiled_matmul, [a, b, c], expected)

## Step 2 — tiling, with a schedule written against the interface

**Tiling** restructures a loop nest into blocks. Outer loops step through the data a tile at a time, and the original computation runs inside on tile-sized slices. It computes exactly the same thing, in a different order.

In session 1 a rewrite like this meant writing C++, compiling a plugin and reloading `mlir-opt`. The **transform dialect** is the other way to do it. A schedule is itself MLIR: a small program whose *input* is your program, run by an interpreter. No C++, no rebuild — edit the file, re-run the cell.

**`session2/tile_linalg.mlir`** is a working schedule; read it before changing anything. It never names an op. It matches `interface{LinalgOp}`, so it finds any Linalg op, and the transformation it applies, `tile_using_for`, is itself written against that interface. One schedule tiles matvec and matmul alike. Handles like `%ops` are not values in your program — they are *references* to ops in it, and transform ops act through them.

It tiles with `[32, 32]`: one tile size per loop, in the order of `iterator_types` above. That covers both of matvec's loops, but only the first two of matmul's three, `i` and `j`. Matmul's `k` loop is not tiled.

**Exercise: extend `tile_linalg.mlir` so that matmul's `k` loop is tiled by 32 as well.** Two things you may need: `transform.structured.match ops{["linalg.matmul"]}` matches by op name instead of by interface, and a tile size of `0` leaves that loop untiled. After each edit, round-trip the schedule, then re-run the matmul cell and the check after it.

In [ ]:
# Opens the file you edit. On Colab, files.view() puts it in the editor pane.
# Running locally there is no google.colab, so this prints the path instead
# and you open it in your own editor.
SCHEDULE = f"{REPO_DIR}/session2/tile_linalg.mlir"

try:
    from google.colab import files

    files.view(SCHEDULE)
except ModuleNotFoundError:
    print(f"open in your editor -- you edit this:\n    {SCHEDULE}")

Round-trip the schedule after every edit, before applying it. With no pass pipeline, `mlir-opt` parses the schedule and runs the verifier on it, so a malformed schedule fails here, right next to your edit, instead of halfway through applying it.

That checks that the schedule is well-formed, not that it does what you meant. Applying it, next, is what checks that.

In [ ]:
!mlir-opt {SCHEDULE}
assert _exit_code == 0

Apply the schedule to matvec first. `--transform-preload-library` loads the schedule, and `--transform-interpreter` finds `@__transform_main` in it and applies it to the file.

Tiling only restructures; it does not lower anything. The result is still Linalg: `scf.for` tile loops stepping by 32, around the original op working on `memref.subview` slices of its operands. Both of matvec's loops come out tiled.

In [ ]:
MATVEC_TILED = "linalg_matvec_tiled.mlir"

!mlir-opt {MATVEC_FILE} \
    --transform-preload-library=transform-library-paths={SCHEDULE} \
    --transform-interpreter \
    -o {MATVEC_TILED}
assert _exit_code == 0
!cat {MATVEC_TILED}

assert "memref.subview" in open(MATVEC_TILED).read(), "@matvec was not tiled"

Now the same schedule, unchanged, on matmul.

In [ ]:
MATMUL_TILED = "linalg_matmul_tiled.mlir"

!mlir-opt {MATMUL_FILE} \
    --transform-preload-library=transform-library-paths={SCHEDULE} \
    --transform-interpreter \
    -o {MATMUL_TILED}
assert _exit_code == 0
!cat {MATMUL_TILED}

**This is the check on the exercise:** matmul tiled in all three loops, so every operand of the innermost `linalg.matmul` is a 32 by 32 slice. Only the innermost op's operands count here.

In [ ]:
import re

matmul_line = next(line for line in open(MATMUL_TILED) if "linalg.matmul" in line)
operand_shapes = re.findall(r"memref<(\d+x\d+)xf32", matmul_line)
print("shapes the innermost linalg.matmul works on:", operand_shapes)
assert operand_shapes and all(shape == "32x32" for shape in operand_shapes), (
    "@matmul is not tiled in all three loops yet -- a 512 in these shapes is a "
    "loop that still runs untiled")

Now run the tiled matmul, through the same numpy check as step 1. Tiling that changed the answer would be a bug, not an optimization.

The lowering needs two more passes this time, for the ops tiling introduced:

- **`expand-strided-metadata`** breaks each `memref.subview` into raw pointer, offset and stride arithmetic. Without it the lowering stops on an `unrealized_conversion_cast` that names nothing useful.
- **`lower-affine`** turns that arithmetic, which comes out as `affine.apply` ops, into plain `arith`.

The cell below adds both to `CPU_LOWERING_PIPELINE`, right after `convert-linalg-to-loops`.

In [ ]:
# Tiling introduced memref.subview: add the two passes that lower it.
CPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(llvm-request-c-wrappers),"
    "func.func(convert-linalg-to-loops),"
    "expand-strided-metadata,"
    "lower-affine,"
    "convert-scf-to-cf,"
    "convert-cf-to-llvm,"
    "convert-arith-to-llvm,"
    "finalize-memref-to-llvm,"
    "convert-func-to-llvm,"
    "reconcile-unrealized-casts"
    ")"
)

tiled_matmul = compile_file(MATMUL_TILED)
run_and_check(tiled_matmul, [a, b, c], expected)

## Step 3 — did matmul get faster?

Same computation, same numbers, same single thread. Time the untiled and tiled matmul and see.

`benchmark` keeps the **fastest** of several runs rather than the average. Colab is a shared machine, so a slow run mostly measures your neighbours.

In [ ]:
import time


def benchmark(name, engine, reps=5):
    best = float("inf")
    for _ in range(reps):
        c[...] = 0
        args = memref_args(a, b, c)
        start = time.perf_counter()
        engine.invoke("matmul", *args)
        best = min(best, time.perf_counter() - start)
    gflops = 2 * M * N * K / best / 1e9  # two float ops per innermost iteration
    print(f"{name:>10}: {best * 1000:8.1f} ms   {gflops:6.2f} GFLOP/s")
    return best


untiled_time = benchmark("untiled", untiled_matmul)
tiled_time = benchmark("tiled", tiled_matmul)
print(f"\nspeedup from tiling: {untiled_time / tiled_time:.1f}x")

### Why so much faster?

The usual explanation for tiling is the cache: small blocks keep the data each one touches in fast memory. Cache *capacity* is not the main factor here — the three matrices together are under 1 MB, small enough to sit largely in cache untiled. What tiling did was reshape the inner loop, and that is what paid off.

What changed is the **machine code LLVM generated**, so look at it. The cell below asks the JIT to dump the object code it produced for each version and counts the floating-point multiply and add instructions in `matmul`:

- **scalar** instructions (`mulss`, `addss`, ...) work on one number at a time;
- **packed** instructions (`mulps`, `addps`, ...) work on a whole vector of 4, 8 or 16 numbers at once, in the `xmm`, `ymm` or `zmm` registers.

In [ ]:
import shutil

# objdump comes with binutils. Install it only where it is missing, so this
# also runs locally, where apt-get would need root.
if shutil.which("objdump"):
    print("objdump is already installed")
else:
    !apt-get install -y -qq binutils > /dev/null
    assert _exit_code == 0

In [ ]:
import shutil
import subprocess


def float_ops(path):
    """JIT-compile `path`, dump its object code, and count the float multiply
    and add instructions in `matmul`: scalar (...ss) and packed (...ps)."""
    engine = compile_file(path, enable_object_dump=True)
    obj = os.path.basename(path).replace(".mlir", ".o")
    engine.dump_to_object_file(obj)
    asm = subprocess.run(["objdump", "-d", "--no-show-raw-insn", obj],
                         capture_output=True, text=True).stdout
    body = asm.split("<matmul>:", 1)[1].split("\n\n", 1)[0]
    ops = re.findall(r"\s(v?(?:mul|add|fmadd\w*)(ss|ps))\s+(\S+)", body)
    scalar = sum(1 for _, kind, _ in ops if kind == "ss")
    packed = [operands for _, kind, operands in ops if kind == "ps"]
    width = {"zmm": 16, "ymm": 8, "xmm": 4}
    lanes = max((w for reg, w in width.items() if any(reg in p for p in packed)), default=0)
    return scalar, len(packed), lanes


if shutil.which("objdump") is None:
    print("objdump is not installed here -- `apt-get install binutils`, or skip this cell.")
else:
    print(f"{'':>8} {'scalar ops':>11} {'packed ops':>11}  vector width")
    for label, path in (("untiled", MATMUL_FILE), ("tiled", MATMUL_TILED)):
        scalar, packed, lanes = float_ops(path)
        print(f"{label:>8} {scalar:>11} {packed:>11}  {f'{lanes} floats at once' if packed else '-'}")

**The speedup is vectorization, and tiling is what made it possible.**

1. **Untiled, the innermost loop is the reduction** `c[i,j] += a[i,k] * b[k,j]` over `k`. A running float sum cannot be vectorized without changing the order of the additions, which changes the answer in its last bits, and LLVM will not do that unless told it may. So the untiled matmul multiplies one pair of numbers at a time.
2. **Tiled, `k` runs a small, constant 32 times inside each tile.** LLVM *unrolls* a loop like that completely: it pastes the body out 32 times as straight-line code, and the `k` loop disappears.
3. **What is left around it, the loop over `j`, is a plain loop whose iterations are independent.** That loop *can* be vectorized, so several `c[i,j]` are computed at once. That is the packed column above.

Locality is part of the story too: with `k` unrolled, the innermost loop is `j`, which walks `b` and `c` with unit stride, and the accumulators for a tile row stay in registers. But that reshaping is what makes the vectorization possible, and it is the vector width that buys the order of magnitude.

Both steps are decisions made by cost models inside LLVM, which is why this session fixes the tile size at 32. With LLVM 22 on every x86 CPU Colab hands out, tiles of 16 or 32 get through both steps; tiles of 24 or 64 do not, and the kernel stays scalar. That is a fact about this compiler version, not a law — and a story about cache capacity alone would never have predicted it.

Two lessons carry over to part 2:

- **A transformation is often an *enabler*.** Tiling did little by itself. It turned the loop nest into a shape a later optimization could use.
- **Measure, then read the generated code.** The textbook explanation would have sent you looking in the wrong place.